# TP : prise en main de PyTorch

Nom : ____________________  
Classe : ____________________

Je reprends ici les notions du support de cours, puis les exercices de régression et de classification. Le but est de comprendre les étapes du calcul, et pas seulement de lancer un réseau.

Support de départ : « Activité pratique Bases de Pytorch.ipynb » fourni en cours.

Exécuter les cellules dans l'ordre. Les résultats et les graphiques seront calculés à l'exécution ; aucune mesure n'est renseignée à l'avance. MNIST nécessite une connexion au premier téléchargement.


## 1. Préparer les bibliothèques

J'utilise PyTorch pour les modèles, Matplotlib pour les graphiques et scikit-learn pour Iris et les métriques. La cellule d'installation est à lancer seulement si les bibliothèques manquent.


In [ ]:
# Décommenter si nécessaire, puis redémarrer le noyau.
# %pip install torch torchvision matplotlib scikit-learn


In [ ]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Version de PyTorch :", torch.__version__)
print("Appareil utilisé :", device)


## 2. Manipuler des tenseurs

Un tenseur peut contenir une seule valeur, un vecteur ou une matrice. Sa forme permet de vérifier que les données sont compatibles avec le modèle. Le produit avec @ est un produit matriciel ; * multiplie élément par élément.


In [ ]:
nombre = torch.tensor(3.0)
vecteur = torch.tensor([1., 2., 3.])
matrice = torch.tensor([[1., 2.], [3., 4.]])
for nom, valeur in [("scalaire", nombre), ("vecteur", vecteur), ("matrice", matrice)]:
    print(nom, "forme :", valeur.shape, "dimensions :", valeur.ndim)
print("Type :", matrice.dtype)

tableau = torch.arange(12).reshape(3, 4)
print(tableau)
print("Deuxième colonne :", tableau[:, 1])
print("Première ligne :", tableau[0])
print("Produit matriciel :", matrice @ matrice)
print("Produit élément par élément :", matrice * matrice)
print("Moyenne :", matrice.mean().item(), "Somme :", matrice.sum().item())
print("Tenseur sur l'appareil choisi :", vecteur.to(device).device)


## 3. Calculer les gradients

Pour y = wx + b et L = (y - 10)², les dérivées sont 2(y - 10)x pour w et 2(y - 10) pour b. Avec x = 2, w = 3 et b = 1, on doit trouver -12 et -6. requires_grad=True indique les paramètres à suivre.


In [ ]:
entree = torch.tensor(2.)
poids = torch.tensor(3., requires_grad=True)
biais = torch.tensor(1., requires_grad=True)
sortie = poids * entree + biais
perte = (sortie - 10).square()
perte.backward()
print("Sortie :", sortie.item(), "Perte :", perte.item())
print("Gradient de w :", poids.grad.item())
print("Gradient de b :", biais.grad.item())
print("Vérification à la main :", 2 * (7 - 10) * 2, 2 * (7 - 10))

# autograd.grad renvoie les dérivées sans remplir les attributs .grad.
p2 = torch.tensor(3., requires_grad=True)
b2 = torch.tensor(1., requires_grad=True)
print("Autre méthode :", torch.autograd.grad((p2 * entree + b2 - 10).square(), (p2, b2)))


## 4. Définir un réseau

nn.Linear applique une transformation affine. Pour un réseau plus grand, je regroupe les couches dans nn.Module. Ici, quatre valeurs en entrée donnent trois scores en sortie. ReLU ajoute une non-linéarité entre les couches. L'appel reseau(entrees) passe par forward.


In [ ]:
couche = nn.Linear(3, 1)
print("Sortie d'une couche :", couche(torch.tensor([[1., 2., 3.]])))
print("Poids :", couche.weight)
print("Biais :", couche.bias)

class PetitReseau(nn.Module):
    def __init__(self):
        super().__init__()
        self.couches = nn.Sequential(
            nn.Linear(4, 16), nn.ReLU(),
            nn.Linear(16, 8), nn.ReLU(), nn.Linear(8, 3)
        )

    def forward(self, entrees):
        return self.couches(entrees)

reseau = PetitReseau()
entrees = torch.randn(5, 4)
print(reseau)
print("Entrée :", entrees.shape, "Sortie :", reseau(entrees).shape)


In [ ]:
axe = torch.linspace(-4, 4, 200)
plt.figure(figsize=(7, 4))
for nom, fonction in [("ReLU", torch.relu), ("Sigmoid", torch.sigmoid), ("Tanh", torch.tanh)]:
    plt.plot(axe.numpy(), fonction(axe).numpy(), label=nom)
plt.title("Comparaison des activations")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## 5. Perte, optimiseur et entraînement

La MSE convient à une régression. CrossEntropyLoss convient à plusieurs classes : elle reçoit les scores bruts et des étiquettes entières, sans Softmax préalable. Pour une classification binaire à une seule sortie, on peut utiliser BCEWithLogitsLoss.

La boucle garde le même ordre : prédiction, perte, remise à zéro des gradients, backward, puis step. Sans remise à zéro, les gradients s'accumulent.


In [ ]:
print("MSE :", nn.MSELoss()(torch.tensor([2., 4.]), torch.tensor([3., 5.])).item())
scores = torch.tensor([[2., 1., 0.5], [0.2, 0.1, 2.]])
print("Entropie croisée :", nn.CrossEntropyLoss()(scores, torch.tensor([0, 2])).item())

x_demo = torch.randn(30, 1)
y_demo = 2 * x_demo + 1
modele_demo = nn.Linear(1, 1)
optimiseur = torch.optim.Adam(modele_demo.parameters(), lr=0.05)
for epoque in range(300):
    modele_demo.train()
    erreur = nn.MSELoss()(modele_demo(x_demo), y_demo)
    optimiseur.zero_grad()
    erreur.backward()
    optimiseur.step()
print("Paramètres appris :", modele_demo.weight.item(), modele_demo.bias.item())


## 6. Modes du modèle et mini-lots

train() active le comportement d'entraînement des couches comme Dropout. eval() change ce comportement, mais ne désactive pas les gradients. Pour prédire, j'ajoute donc no_grad(). DataLoader découpe les données en lots et peut les mélanger à chaque passage.


In [ ]:
dropout = nn.Dropout(0.5)
valeurs = torch.ones(10)
dropout.train()
print("Pendant l'entraînement :", dropout(valeurs))
dropout.eval()
with torch.no_grad():
    print("Pendant l'évaluation :", dropout(valeurs))

jeu_demo = TensorDataset(torch.randn(100, 3), torch.randn(100, 1))
lots_demo = DataLoader(jeu_demo, batch_size=16, shuffle=True)
x_lot, y_lot = next(iter(lots_demo))
print("Formes d'un lot :", x_lot.shape, y_lot.shape)


## 7. Régression sur des points simulés

Je génère des points autour de y = 3x + 2 avec un peu de bruit. La droite apprise doit se rapprocher de cette relation. Le R² ci-dessous est calculé sur les données d'entraînement : il décrit l'ajustement, pas la capacité de généralisation.


In [ ]:
torch.manual_seed(42)
x_reg = torch.linspace(-2, 2, 120).reshape(-1, 1)
y_reg = 3 * x_reg + 2 + 0.6 * torch.randn_like(x_reg)
regression = nn.Linear(1, 1)
opt_reg = torch.optim.Adam(regression.parameters(), lr=0.05)
pertes_reg = []
for epoque in range(300):
    regression.train()
    perte = nn.MSELoss()(regression(x_reg), y_reg)
    opt_reg.zero_grad()
    perte.backward()
    opt_reg.step()
    pertes_reg.append(perte.item())
regression.eval()
with torch.no_grad():
    estimation = regression(x_reg)
    r2 = 1 - (y_reg - estimation).square().sum() / (y_reg - y_reg.mean()).square().sum()
print("Poids :", regression.weight.item(), "Biais :", regression.bias.item())
print("R² d'entraînement :", r2.item())
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(pertes_reg)
axes[0].set(title="Apprentissage", xlabel="Époque", ylabel="MSE")
axes[1].scatter(x_reg.numpy(), y_reg.numpy(), s=12, label="Points")
axes[1].plot(x_reg.numpy(), estimation.numpy(), color="red", label="Droite apprise")
axes[1].legend()
plt.tight_layout()
plt.show()


## 8. Classification de deux groupes

Le réseau reçoit deux coordonnées et renvoie deux scores. argmax choisit la classe dont le score est le plus grand. Les métriques de cet exemple sont calculées sur l'entraînement ; les exercices suivants utilisent un test séparé.


In [ ]:
torch.manual_seed(0)
groupe0 = torch.randn(120, 2) + torch.tensor([-2., 0.])
groupe1 = torch.randn(120, 2) + torch.tensor([2., 0.])
x_cls = torch.cat((groupe0, groupe1))
y_cls = torch.cat((torch.zeros(120), torch.ones(120))).long()

class Classifieur2D(nn.Module):
    def __init__(self):
        super().__init__()
        self.couches = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 16), nn.ReLU(), nn.Linear(16, 2))

    def forward(self, x):
        return self.couches(x)

classifieur = Classifieur2D()
opt_cls = torch.optim.Adam(classifieur.parameters(), lr=0.01)
for epoque in range(250):
    classifieur.train()
    perte = nn.CrossEntropyLoss()(classifieur(x_cls), y_cls)
    opt_cls.zero_grad()
    perte.backward()
    opt_cls.step()
classifieur.eval()
with torch.no_grad():
    classes = classifieur(x_cls).argmax(1)
print(classification_report(y_cls.numpy(), classes.numpy(), zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_cls.numpy(), classes.numpy())
plt.show()

xx, yy = torch.meshgrid(
    torch.linspace(x_cls[:, 0].min() - 1, x_cls[:, 0].max() + 1, 150),
    torch.linspace(x_cls[:, 1].min() - 1, x_cls[:, 1].max() + 1, 150), indexing="ij"
)
with torch.no_grad():
    zones = classifieur(torch.stack((xx.flatten(), yy.flatten()), dim=1)).argmax(1).reshape(xx.shape)
plt.contourf(xx.numpy(), yy.numpy(), zones.numpy(), alpha=0.25)
plt.scatter(x_cls[:, 0], x_cls[:, 1], c=y_cls, s=12)
plt.title("Frontière de décision")
plt.show()

torch.save(classifieur.state_dict(), "classifieur_2d_tp.pt")
copie = Classifieur2D()
copie.load_state_dict(torch.load("classifieur_2d_tp.pt", map_location="cpu", weights_only=True))
copie.eval()
with torch.no_grad():
    print("Rechargement correct :", torch.equal(classes, copie(x_cls).argmax(1)))


## 9. Exercice : estimer un salaire

Les données sont fictives. Le salaire dépend de l'ancienneté et d'une note d'expertise. Je sépare 80 % des données pour apprendre et 20 % pour tester. La normalisation est calculée uniquement sur l'entraînement et les salaires sont exprimés en milliers de DH pendant le calcul.


In [ ]:
torch.manual_seed(42)
anciennete = torch.randint(0, 21, (300, 1)).float()
expertise = torch.randint(1, 6, (300, 1)).float()
x_salaire = torch.cat((anciennete, expertise), dim=1)
y_salaire = 4000 + 600 * anciennete + 1500 * expertise + 500 * torch.randn(300, 1)
print("Dimensions :", x_salaire.shape, y_salaire.shape)
print("Salaire moyen :", round(y_salaire.mean().item(), 2), "DH")
plt.scatter(anciennete.flatten(), y_salaire.flatten(), c=expertise.flatten(), s=15)
plt.colorbar(label="Expertise")
plt.xlabel("Ancienneté (années)")
plt.ylabel("Salaire (DH)")
plt.show()

ordre = torch.randperm(300)
indices_train, indices_test = ordre[:240], ordre[240:]
x_train = x_salaire[indices_train]
x_test = x_salaire[indices_test]
y_train = y_salaire[indices_train] / 1000
y_test = y_salaire[indices_test] / 1000
moyenne = x_train.mean(0, keepdim=True)
ecart = x_train.std(0, keepdim=True)
x_train_norm = (x_train - moyenne) / ecart
x_test_norm = (x_test - moyenne) / ecart


In [ ]:
modele_salaire = nn.Linear(2, 1)
opt_salaire = torch.optim.SGD(modele_salaire.parameters(), lr=0.05)
historique = []
for epoque in range(500):
    modele_salaire.train()
    predictions = modele_salaire(x_train_norm)
    perte = nn.MSELoss()(predictions, y_train)
    opt_salaire.zero_grad()
    perte.backward()
    opt_salaire.step()
    historique.append(perte.item())
    if (epoque + 1) % 100 == 0:
        print(f"Époque {epoque + 1}, MSE : {perte.item():.4f}")
plt.plot(historique)
plt.xlabel("Époque")
plt.ylabel("MSE (milliers de DH au carré)")
plt.show()

modele_salaire.eval()
with torch.no_grad():
    salaires_estimes = modele_salaire(x_test_norm) * 1000
    mae = (salaires_estimes - y_test * 1000).abs().mean()
    nouveau = (torch.tensor([[7., 4.]]) - moyenne) / ecart
    salaire_nouveau = modele_salaire(nouveau) * 1000
    coefficients = modele_salaire.weight * 1000 / ecart
    constante = modele_salaire.bias * 1000 - (coefficients * moyenne).sum()
print(f"MAE sur le test : {mae.item():.2f} DH")
print(f"Salaire pour 7 ans et expertise 4 : {salaire_nouveau.item():.2f} DH")
print("Valeur théorique sans bruit :", 4000 + 600 * 7 + 1500 * 4, "DH")
print("Coefficients en DH :", coefficients.flatten().tolist())
print("Constante en DH :", constante.item())


### Réponses aux questions

À expertise constante, une année supplémentaire ajoute 600 DH dans la règle de génération. La perte ne sera pas nulle à cause du bruit aléatoire. Si ce bruit augmente, les salaires deviennent plus difficiles à prévoir avec ces deux variables.

Pour 40 ans d'expérience, le modèle extrapole car il n'a vu que des anciennetés de 0 à 20 ans. Une telle prédiction doit être prise avec prudence. Enfin, traiter l'expertise comme un nombre suppose un même effet entre deux niveaux successifs.


## 10. Exercice : reconnaître les fleurs Iris

Les quatre entrées sont la longueur et la largeur du sépale, puis celles du pétale. Les étiquettes sont de type long car CrossEntropyLoss attend des indices de classes. Le scaler est ajusté sur l'entraînement pour éviter de transmettre des informations du test au modèle.


In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

iris = load_iris()
x_appr, x_eval, y_appr, y_eval = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42, stratify=iris.target
)
scaler = StandardScaler()
x_appr = torch.tensor(scaler.fit_transform(x_appr), dtype=torch.float32)
x_eval = torch.tensor(scaler.transform(x_eval), dtype=torch.float32)
y_appr = torch.tensor(y_appr, dtype=torch.long)
y_eval = torch.tensor(y_eval, dtype=torch.long)
print(x_appr.shape, y_appr.shape)

def apprendre_iris(modele):
    opt = torch.optim.Adam(modele.parameters(), lr=0.01)
    pertes = []
    for epoque in range(200):
        modele.train()
        perte = nn.CrossEntropyLoss()(modele(x_appr), y_appr)
        opt.zero_grad()
        perte.backward()
        opt.step()
        pertes.append(perte.item())
    modele.eval()
    with torch.no_grad():
        predictions = modele(x_eval).argmax(1)
    return pertes, predictions

torch.manual_seed(42)
modele_iris = nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 3))
pertes_iris, pred_iris = apprendre_iris(modele_iris)
torch.manual_seed(42)
modele_lineaire = nn.Linear(4, 3)
pertes_lineaires, pred_lineaires = apprendre_iris(modele_lineaire)
print("Accuracy réseau :", (pred_iris == y_eval).float().mean().item())
print("Accuracy linéaire :", (pred_lineaires == y_eval).float().mean().item())
print(classification_report(y_eval.numpy(), pred_iris.numpy(), target_names=iris.target_names, zero_division=0))
plt.plot(pertes_iris, label="Avec couche cachée")
plt.plot(pertes_lineaires, label="Sans couche cachée")
plt.xlabel("Époque")
plt.ylabel("Perte d'entraînement")
plt.legend()
plt.show()
ConfusionMatrixDisplay.from_predictions(y_eval.numpy(), pred_iris.numpy(), display_labels=iris.target_names)
plt.show()
with torch.no_grad():
    fleur = torch.tensor(scaler.transform([[5.1, 3.5, 1.4, 0.2]]), dtype=torch.float32)
    espece = modele_iris(fleur).argmax(1).item()
print("Espèce prédite :", iris.target_names[espece])


Le jeu de test ne contient que 30 fleurs. Une différence de score sur ce petit ensemble ne suffit donc pas à conclure qu'une architecture est toujours meilleure. Je garde les deux scores pour comparer les résultats obtenus à l'exécution.


## 11. Exercice : classifier MNIST avec un réseau dense

Chaque image mesure 28 × 28 pixels et possède un seul canal. Un lot a donc la forme [nombre d'images, 1, 28, 28]. Je mélange l'entraînement pour varier les lots. Le test garde un ordre fixe.

Le modèle dense apprend pendant cinq époques. Les fonctions suivantes servent aussi au CNN. Les métriques sont calculées sur tout le test ; les moyennes macro donnent le même poids à chaque chiffre.


In [ ]:
from torchvision import datasets, transforms

conversion = transforms.ToTensor()
mnist_train = datasets.MNIST("./data", train=True, download=True, transform=conversion)
mnist_test = datasets.MNIST("./data", train=False, download=True, transform=conversion)
lots_train = DataLoader(mnist_train, batch_size=64, shuffle=True)
lots_test = DataLoader(mnist_test, batch_size=256, shuffle=False)

def entrainer_chiffres(modele, lots, epoques):
    opt = torch.optim.Adam(modele.parameters(), lr=0.001)
    historique = []
    for epoque in range(epoques):
        modele.train()
        somme = 0.0
        nombre = 0
        for images, etiquettes in lots:
            images, etiquettes = images.to(device), etiquettes.to(device)
            scores = modele(images)
            perte = nn.CrossEntropyLoss()(scores, etiquettes)
            opt.zero_grad()
            perte.backward()
            opt.step()
            somme += perte.item() * images.size(0)
            nombre += images.size(0)
        historique.append(somme / nombre)
        print(f"Époque {epoque + 1}/{epoques} : {historique[-1]:.4f}")
    return historique

@torch.no_grad()
def evaluer_chiffres(modele, lots, titre, normalise=False):
    modele.eval()
    vraies, predites, erreurs = [], [], []
    for images, etiquettes in lots:
        predictions = modele(images.to(device)).argmax(1).cpu()
        vraies.extend(etiquettes.tolist())
        predites.extend(predictions.tolist())
        for i in (predictions != etiquettes).nonzero(as_tuple=True)[0].tolist():
            if len(erreurs) < 10:
                erreurs.append((images[i, 0], etiquettes[i].item(), predictions[i].item()))
    print(titre)
    print(classification_report(vraies, predites, labels=list(range(10)), digits=4, zero_division=0))
    ConfusionMatrixDisplay.from_predictions(vraies, predites, labels=list(range(10)))
    plt.title(titre)
    plt.show()
    if erreurs:
        fig, axes = plt.subplots(1, len(erreurs), figsize=(2 * len(erreurs), 2), squeeze=False)
        for ax, (image, vraie, prediction) in zip(axes.flat, erreurs):
            if normalise:
                image = image * 0.3081 + 0.1307
            ax.imshow(image.numpy(), cmap="gray", vmin=0, vmax=1)
            ax.set_title(f"Vrai {vraie} / prédit {prediction}")
            ax.axis("off")
        plt.tight_layout()
        plt.show()
    else:
        print("Aucune erreur sur ce test.")
    return torch.tensor(predites)

def construire_dense():
    return nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10))

dense = construire_dense().to(device)
pertes_dense = entrainer_chiffres(dense, lots_train, epoques=5)
pred_dense = evaluer_chiffres(dense, lots_test, "MNIST : réseau dense")
plt.plot(pertes_dense)
plt.xlabel("Époque")
plt.ylabel("Perte moyenne")
plt.show()

torch.save(dense.state_dict(), "mnist_dense_tp.pt")
dense_recharge = construire_dense().to(device)
dense_recharge.load_state_dict(torch.load("mnist_dense_tp.pt", map_location=device, weights_only=True))
dense_recharge.eval()
dense.eval()
with torch.no_grad():
    exemple = next(iter(lots_test))[0].to(device)
    print("Sorties identiques après rechargement :", torch.allclose(dense(exemple), dense_recharge(exemple)))


## 12. Exercice : classifier MNIST avec un CNN

Les convolutions utilisent les voisins d'un pixel. Je garde l'architecture demandée : 1 → 16 → 32 canaux, puis deux couches linéaires. Après les deux MaxPool, la taille passe de 28 à 14 puis à 7. Il reste donc 32 × 7 × 7 valeurs avant la couche dense.

J'ajoute le Dropout demandé avant la dernière couche. Les données sont normalisées pour l'apprentissage et remises à leur échelle d'origine pour l'affichage.


In [ ]:
normalisation = transforms.Compose([
    transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))
])
cnn_train = datasets.MNIST("./data", train=True, download=True, transform=normalisation)
cnn_test = datasets.MNIST("./data", train=False, download=True, transform=normalisation)
lots_cnn_train = DataLoader(cnn_train, batch_size=64, shuffle=True)
lots_cnn_test = DataLoader(cnn_test, batch_size=256, shuffle=False)
images, etiquettes = next(iter(lots_cnn_train))
print("Images :", images.shape, "Étiquettes :", etiquettes.shape)
fig, axes = plt.subplots(1, 6, figsize=(10, 2))
for i, ax in enumerate(axes):
    ax.imshow(images[i, 0] * 0.3081 + 0.1307, cmap="gray", vmin=0, vmax=1)
    ax.set_title(str(etiquettes[i].item()))
    ax.axis("off")
plt.tight_layout()
plt.show()

class CNNChiffres(nn.Module):
    def __init__(self):
        super().__init__()
        self.convolutions = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.classification = nn.Sequential(
            nn.Flatten(), nn.Linear(32 * 7 * 7, 128), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.classification(self.convolutions(x))

cnn = CNNChiffres().to(device)
with torch.no_grad():
    sortie = cnn(images.to(device))
assert sortie.shape == (images.size(0), 10)
print("Sortie du CNN :", sortie.shape)
pertes_cnn = entrainer_chiffres(cnn, lots_cnn_train, epoques=3)
pred_cnn = evaluer_chiffres(cnn, lots_cnn_test, "MNIST : CNN", normalise=True)
plt.plot(pertes_cnn)
plt.xlabel("Époque")
plt.ylabel("Perte moyenne")
plt.show()
torch.save(cnn.state_dict(), "mnist_cnn_tp.pth")
cnn_recharge = CNNChiffres().to(device)
cnn_recharge.load_state_dict(torch.load("mnist_cnn_tp.pth", map_location=device, weights_only=True))
cnn_recharge.eval()
cnn.eval()
with torch.no_grad():
    entree = images.to(device)
    print("Rechargement du CNN :", torch.allclose(cnn(entree), cnn_recharge(entree)))


## Bilan

Les exemples utilisent les mêmes étapes : préparer les tenseurs, définir le modèle, choisir une perte, calculer les gradients et mettre à jour les paramètres. Pour une évaluation correcte, il faut aussi séparer le test de l'entraînement et réutiliser les mêmes transformations.

Après exécution, je peux comparer la MAE des salaires, les deux modèles Iris et les résultats MNIST. La baisse de la perte d'entraînement ne suffit pas à elle seule à prouver qu'un modèle fonctionne bien sur de nouvelles données.
